In [ ]:
# ==============================================================
# DAILY BIAS CORRECTION (QM)
# Final = (CHIRPS × Grid) → Scaled to monthly corrected rasters
# ==============================================================
import os
from glob import glob
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import rasterio
from scipy.ndimage import gaussian_filter

# -------------------------- PATHS --------------------------
BASE = r"D:\Working_Directory"
csv_dir       = os.path.join(BASE, "Timeseries")
tif_dir       = os.path.join(BASE, "Daily_CHIRPS")
monthly_dir   = os.path.join(BASE, "monthly_Corrected")
coords_path   = os.path.join(BASE, "Coordinates.csv")

output_root = os.path.join(BASE, "OUTPUT_QM")
inter_dir   = os.path.join(output_root, "intermediate")
final_dir   = os.path.join(output_root, "final_corrected")

os.makedirs(inter_dir, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)

# -------------------------- LOAD COORDS --------------------------
coords_df = pd.read_csv(coords_path)

# -------------------------- 1. REFERENCE RASTER --------------------------
ref = glob(os.path.join(tif_dir, "*.tif"))[0]
with rasterio.open(ref) as src:
    meta = src.meta.copy()
    H, W = src.height, src.width
    tr = src.transform

cols, rows = np.meshgrid(np.arange(W), np.arange(H))
xs, ys = rasterio.transform.xy(tr, rows, cols)
grid_pts = np.column_stack([np.ravel(xs), np.ravel(ys)])

# -------------------------- 2. IDW --------------------------
def idw(coords, values, targets, power=2):
    dist = np.sqrt(((targets[:, None, :] - coords[None, :, :])**2).sum(axis=2))
    dist[dist == 0] = 1e-12
    w = 1.0 / dist**power
    return ((w * values[None, :]).sum(axis=1) / w.sum(axis=1)).astype(np.float32)

# -------------------------- 3. QM FUNCTION --------------------------
def quantile_mapping(obs, sim):
    obs_sorted = np.sort(obs)
    ranks = sim.argsort().argsort()
    q = ranks / (len(sim) - 1)
    return np.quantile(obs_sorted, q)

# -------------------------- 4. MONTHLY FACTORS --------------------------
monthly_cfs = defaultdict(dict)

for fp in glob(os.path.join(csv_dir, "*.csv")):
    st = os.path.splitext(os.path.basename(fp))[0]

    df = pd.read_csv(fp, parse_dates=["Date"])
    df = df.dropna(subset=["InSitu", "CHIRPS"])
    df["Month"] = df["Date"].dt.month

    for m in range(1, 13):
        sub = df[df["Month"] == m]

        if len(sub) < 3:
            monthly_cfs[st][m] = 1.0
            continue

        obs = sub["InSitu"].values
        sim = sub["CHIRPS"].values

        corrected = quantile_mapping(obs, sim)

        ratio = corrected / sim
        ratio = ratio[np.isfinite(ratio) & (sim > 0)]

        factor = np.median(ratio) if len(ratio) > 0 else 1.0
        monthly_cfs[st][m] = factor

# Save factors
pd.DataFrame([
    {"Station": s, "Month": m, "Factor": f}
    for s in monthly_cfs for m, f in monthly_cfs[s].items()
]).to_csv(os.path.join(output_root, "monthly_factors.csv"), index=False)

# -------------------------- 5. INTERMEDIATE RASTERS --------------------------
tif_files = sorted(glob(os.path.join(tif_dir, "*.tif")))

for tif_path in tif_files:
    fname = os.path.basename(tif_path)
    date_str = fname.split("_")[-1].replace(".tif", "")
    dt = datetime.strptime(date_str, "%Y.%m.%d")
    month = dt.month

    stations = []
    factors = []

    for st in monthly_cfs:
        stations.append(st)
        factors.append(monthly_cfs[st][month])

    sub = coords_df[coords_df["Station"].isin(stations)]
    coords = np.column_stack([sub["Longitude"], sub["Latitude"]])
    values = np.array(factors)

    factor_grid = idw(coords, values, grid_pts).reshape((H, W))
    factor_grid = gaussian_filter(factor_grid, sigma=1.5)

    with rasterio.open(tif_path) as src:
        chirps = src.read(1).astype(np.float32)
        corrected = chirps * factor_grid
        meta = src.meta.copy()

    out = os.path.join(inter_dir, fname.replace(".tif", "_inter.tif"))
    with rasterio.open(out, "w", **meta) as dst:
        dst.write(corrected, 1)

# -------------------------- 6. MONTHLY ANCHORING --------------------------
monthly_files = glob(os.path.join(monthly_dir, "*_corrected_*.tif"))
monthly_map = {}

for p in monthly_files:
    ds = os.path.basename(p).split("_")[-1].replace(".tif", "")
    y, m = map(int, ds.split(".")[:2])
    monthly_map[(y, m)] = p

daily_groups = defaultdict(list)

for fp in glob(os.path.join(inter_dir, "*_inter.tif")):
    parts = os.path.basename(fp).split("_")
    y, m, d = map(int, parts[-3:])
    daily_groups[(y, m)].append(fp)

for (y, m), files in daily_groups.items():
    mon_path = monthly_map.get((y, m))
    if not mon_path:
        continue

    with rasterio.open(mon_path) as src:
        mon_total = src.read(1)

    daily_sum = np.zeros_like(mon_total)

    for f in files:
        with rasterio.open(f) as src:
            daily_sum += src.read(1)

    scale = np.ones_like(mon_total)
    mask = daily_sum > 0
    scale[mask] = mon_total[mask] / daily_sum[mask]

    for f in files:
        with rasterio.open(f) as src:
            arr = src.read(1)
            meta = src.meta.copy()

        final = arr * scale

        out = os.path.join(final_dir, os.path.basename(f).replace("_inter", "_corrected"))
        with rasterio.open(out, "w", **meta) as dst:
            dst.write(final, 1)

print("DAILY QM (MONTHLY FACTOR MODE) COMPLETED")